# k-Nearest Neighbors classifier tutorial

This notebook trains `mlpackage.supervised_learning.KNeighborsClassifier` on the **Wine** dataset: three Italian cultivar classes described by **13 continuous chemical measurements**. Features span different scales, so we **standardize using training statistics only**, then evaluate on a held-out test set.

## Prerequisites and goals

**You will**
- Load multiclass labels with numeric features.
- Split into train/test with stratification.
- Fit `StandardScaler` on **training** rows and transform train/test (avoid leakage).
- Fit lazy KNN storage and **vote over Euclidean neighbors** (distance and vote formulas in Steps 3–4).
- Measure accuracy, confusion counts, and compare **`n_neighbors`** on the same split.
- Visualize two standardized features colored by class.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, package `mlpackage`.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from mlpackage.supervised_learning import KNeighborsClassifier

## Step 1: Load Wine

The UCI Wine dataset bundled in scikit-learn has **178** wines and **13** numeric descriptors (alcohol, phenols, color intensity, …). Class labels encode three cultivars. Unequal class counts make **stratified splitting** helpful.

In [ ]:
wine = load_wine()
X = np.asarray(wine.data, dtype=float)
y = np.asarray(wine.target)

feature_names = list(wine.feature_names)
target_names = list(wine.target_names)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", feature_names)
print("Classes:", {i: target_names[i] for i in range(len(target_names))})
print("\nLabel counts:", dict(zip(*np.unique(y, return_counts=True))))
print("\nFirst row (raw):", X[0])
print("Feature ranges (min/max per column):")
print(np.vstack([X.min(axis=0), X.max(axis=0)]).round(2))

## Step 2: Stratified train/test split

Hold out test rows **before** any scaling or distance computations so reported accuracy reflects **out-of-sample** generalization. **`stratify=y`** preserves each class fraction in both folds.

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape[0], "  Test:", X_test.shape[0])
print("Train counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test counts :", dict(zip(*np.unique(y_test, return_counts=True))))

## Step 3: Standardize features (train-only fit)

KNN here uses **Euclidean distance**. If one feature spans a huge numeric range and another is tiny, distance is dominated by the large-scale coordinate. **`StandardScaler`** subtracts the **training** mean and divides by the **training** standard deviation per feature, then transforms train and test so test rows never influence the scale.

### Euclidean distance (this implementation)

Between vectors $\mathbf{x}, \mathbf{z} \in \mathbb{R}^p$,

$$
d(\mathbf{x}, \mathbf{z}) = \sqrt{\sum_{j=1}^{p}(x_j - z_j)^2}.
$$

Because $d$ squares coordinate differences, **comparable feature scales** matter unless all columns are already on similar units.



In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train standardized shape:", X_train_s.shape)
print("Quick check — train column means (≈0):", X_train_s.mean(axis=0).round(6)[:5], "...")
print("Quick check — train column stds (≈1):", X_train_s.std(axis=0, ddof=0).round(3)[:5], "...")

## Step 4: Fit `KNeighborsClassifier`

`fit` **stores** the training matrix and labels—there are no learned weights. Hyperparameter **`n_neighbors`** controls bias/variance: small $k$ tracks local noise; large $k$ smooths decision regions.

### Majority vote

For the $k$ nearest neighbors with labels $y_{(1)},\ldots,y_{(k)}$, this code predicts

$$
\hat{y} = \arg\max_{c} \sum_{\ell=1}^{k} \mathbf{1}\{ y_{(\ell)} = c \}.
$$

With $k=1$, decision regions are **Voronoi** cells around training points. For larger $k$, predictions are **local plurality** votes; distance is still Euclidean in the **scaled** space from Step 3.



In [ ]:
N_NEIGHBORS = 5

knn = KNeighborsClassifier(n_neighbors=N_NEIGHBORS)
knn.fit(X_train_s, y_train)

print("Training samples (stored by fit):", X_train_s.shape[0])
print("Using n_neighbors =", knn.n_neighbors)

## Step 5: Predict on the held-out test set

Each test row receives the majority label among its **`n_neighbors`** nearest **scaled** training vectors.

In [ ]:
y_pred = knn.predict(X_test_s)

print("First 15 predictions:", y_pred[:15].tolist())
print("First 15 true labels:", y_test[:15].tolist())

## Step 6: Accuracy, confusion matrix, per-class hits

`score` returns overall accuracy. The package also exposes `confusion_matrix` for class-by-class mistakes.

In [ ]:
acc_train = knn.score(X_train_s, y_train)
acc_test = knn.score(X_test_s, y_test)

print(f"Train accuracy: {acc_train:.4f}")
print(f"Test accuracy : {acc_test:.4f}")

cm = knn.confusion_matrix(X_test_s, y_test)
display(cm)

for klass in sorted(np.unique(y)):
    mask = y_test == klass
    hits = int(np.sum((y_pred == y_test) & mask))
    total = int(np.sum(mask))
    print(f"Class {klass} ({target_names[klass]}): {hits}/{total} correct")

## Step 7: Visualize two standardized features

We plot **`alcohol`** vs **`color_intensity`** (columns 0 and 9 in sklearn's ordering) after scaling all rows with the **same** scaler—geometry mirrors what Euclidean distance sees along those axes (the full model still uses all 13 dimensions).

In [ ]:
X_all_s = scaler.transform(X)

i, j = 0, 9
colors = ["tab:orange", "tab:green", "tab:blue"]

plt.figure(figsize=(7, 5))
for klass in sorted(np.unique(y)):
    m = y == klass
    plt.scatter(
        X_all_s[m, i],
        X_all_s[m, j],
        c=colors[klass],
        label=target_names[klass],
        alpha=0.85,
        edgecolors="black",
        linewidths=0.3,
        s=44,
    )
plt.xlabel(f"{feature_names[i]} (standardized)")
plt.ylabel(f"{feature_names[j]} (standardized)")
plt.title("Wine: two features after StandardScaler")
plt.legend()
plt.tight_layout()

from pathlib import Path

_nb_here = Path("k_neighbors_classifier_tutorial.ipynb")
if _nb_here.is_file():
    _plot_path = _nb_here.with_name("wine_two_features_scatter.png")
else:
    _plot_path = Path(
        "examples/supervised_learning/k_neighbors_classifier/wine_two_features_scatter.png"
    )
    _plot_path.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_plot_path.resolve()}")
plt.show()

## Step 8: Compare `n_neighbors` on the same split

Reuse **`X_train_s` / `X_test_s`** so only \(k\) changes—fair comparison of capacity.

In [ ]:
k_values = [1, 3, 5, 7, 9, 15]
rows = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_s, y_train)
    rows.append(
        {
            "n_neighbors": k,
            "train_acc": model.score(X_train_s, y_train),
            "test_acc": model.score(X_test_s, y_test),
        }
    )

summary = pd.DataFrame(rows)
display(summary.round(4))